# 09 - Contributor Funnel

Build contributor engagement funnels (1st PR, 2nd, 5th, 10th, regular),
compute stage-over-stage retention rates, and compare funnels across
segments (language, org type).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from oss_pulse.analyze.funnel import (
    FUNNEL_STAGES,
    build_contributor_funnel,
    compute_retention_rates,
    funnel_by_segment,
)
from oss_pulse.visualize.comparison import plot_funnel
from oss_pulse.visualize.style import PALETTE, setup_style

setup_style()

In [ ]:
# Load featured PR data with repo metadata
DATA_DIR = Path("../data/processed")
RAW_DIR = Path("../data/raw")

pr_df = pd.read_parquet(DATA_DIR / "pr_events_featured.parquet")
repos_df = pd.read_parquet(RAW_DIR / "top_repos.parquet")

# Enrich with language for segment analysis
pr_enriched = pr_df.merge(
    repos_df[["repo_name", "language", "org_type"]],
    on="repo_name",
    how="left",
)

print(f"PR events: {len(pr_enriched)} rows")
print(f"Human authors: {(~pr_enriched['is_bot']).sum()}")
print(f"Funnel stages: {FUNNEL_STAGES}")

In [ ]:
# Overall contributor funnel
# TODO: run with real data
funnel = build_contributor_funnel(pr_enriched)
print("Overall Contributor Funnel:")
funnel

In [ ]:
# Retention rates between stages
# TODO: run with real data
funnel_with_retention = compute_retention_rates(funnel)
print("Funnel with retention rates:")
funnel_with_retention

In [ ]:
# Visualise the funnel
# TODO: run with real data
fig = plot_funnel(funnel, title="Overall Contributor Funnel")
fig.show()

In [ ]:
# Funnel by language
# TODO: run with real data
lang_funnel = funnel_by_segment(pr_enriched, segment="language")
print("Funnel by language:")
lang_funnel.head(15)

# Side-by-side retention comparison
fig, ax = plt.subplots(figsize=(12, 6))
for lang in lang_funnel["language"].unique()[:5]:
    subset = lang_funnel[lang_funnel["language"] == lang]
    ax.plot(subset["stage"], subset["percentage"], marker="o", linewidth=1.5, label=lang)

ax.set_title("Contributor Funnel by Language", fontsize=14, fontweight="bold")
ax.set_xlabel("Funnel Stage")
ax.set_ylabel("% of Contributors")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Funnel by org type
# TODO: run with real data
org_funnel = funnel_by_segment(pr_enriched, segment="org_type")
print("Funnel by org type:")
org_funnel

fig, ax = plt.subplots(figsize=(10, 6))
colors = [PALETTE["primary"], PALETTE["success"], PALETTE["warning"]]
for i, org in enumerate(org_funnel["org_type"].unique()):
    subset = org_funnel[org_funnel["org_type"] == org]
    ax.plot(
        subset["stage"], subset["percentage"],
        marker="s", linewidth=1.5, color=colors[i % len(colors)], label=org,
    )

ax.set_title("Contributor Funnel by Org Type", fontsize=14, fontweight="bold")
ax.set_xlabel("Funnel Stage")
ax.set_ylabel("% of Contributors")
ax.legend()
plt.tight_layout()
plt.show()